I will arrange the new code into a cell-based format, suitable for a Jupyter notebook. I've separated the code into logical blocks for imports, configuration, data loading, helper functions, core processing functions, feature extraction, visualization, and the main execution loop. This makes the code easier to read, understand, and run sequentially in a notebook environment.

-----

### **Cell 1: Imports**

This cell contains all the necessary library imports, consolidated at the beginning.

In [1]:
import os
import numpy as np
import pandas as pd
import librosa
import parselmouth
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from adjustText import adjust_text
from matplotlib.cm import get_cmap

### **Cell 2: Configuration and File Paths**

This cell defines all the global parameters and file paths, making it easy to modify them.

In [ ]:
# --- File Paths and Directories ---
file_path = r"C:\Users\admin\.spyder-py3\learnings\roots pro\songs 01.xlsx"
sep_audio_dir = r"C:\Users\admin\.spyder-py3\learnings\roots pro\tools\separated\vocals"
output_csv = r"C:\Users\admin\.spyder-py3\learnings\roots pro\song_features.csv"

# --- Audio and Processing Parameters ---
sr = 16000
sr_nov = 100  # Note: `sr_nov` seems unused but kept for context.
controls = {
    "time_step": 0.01,
    "pitch_floor": 75.0,
    "max_number_of_candidates": 15,
    "very_accurate": False,
    "silence_threshold": 0.03,
    "voicing_threshold": 0.35,
    "octave_cost": 0.1,
    "octave_jump_cost": 0.5,
    "voiced_unvoiced_cost": 0.4,
    "pitch_ceiling": 600
}

### **Cell 3: Data Loading**

This cell loads the Excel data and finds all the relevant audio files to process.

In [ ]:
# Load all sheets from the Excel file into a single DataFrame
xlsx = pd.ExcelFile(file_path)
dfs = [pd.read_excel(xlsx, sheet_name=s).assign(SheetName=s) for s in xlsx.sheet_names]
final_df = pd.concat(dfs, ignore_index=True)
print("Data loaded successfully. Shape:", final_df.shape)
print(final_df.head())

# Get list of audio files to process
song_list = [f for f in os.listdir(sep_audio_dir) if f.endswith(".wav")]
print(f"Found {len(song_list)} audio files")

### **Cell 4: Helper and Core Processing Functions**

This cell groups together the helper function (`force_1d`) and the main processing functions (`audio_to_sine_wave`, `hz_to_cents`, `remove_transition_regions_scrollable`).

In [ ]:
def force_1d(x, length=None):
    """Convert anything to a 1D numpy array with proper error handling."""
    try:
        x = np.atleast_1d(x).astype(float)
        if length is not None:
            if x.size == 0:
                return np.zeros(length)
            elif x.size != length:
                return np.resize(x, length)
        return x
    except Exception as e:
        print(f"Error in force_1d: {e}")
        return np.zeros(length) if length else np.array([])

def audio_to_sine_wave(filepath, sr=sr):
    """Extract pitch from audio file with error handling."""
    try:
        snd = parselmouth.Sound(filepath).resample(sr)
        pitch = snd.to_pitch_ac(**controls)
        f0 = pitch.selected_array['frequency']  # Hz
        times = np.array([i/100 for i in range(len(f0))])
        voiced_regions = np.sign(f0)
        return f0, times, voiced_regions
    except Exception as e:
        print(f"Error in audio_to_sine_wave for {filepath}: {e}")
        return np.array([]), np.array([]), np.array([])

def hz_to_cents(f0, name, final_df):
    """Convert Hz to cents with robust tonic handling."""
    try:
        tonic_str = final_df.loc[final_df['Name'] == name, 'Tonic'].values
        tonic = librosa.note_to_hz(tonic_str[0]) if tonic_str.size > 0 else None

        if tonic is None or tonic == 0:
            voiced_f0 = f0[f0 > 0]
            tonic = np.median(voiced_f0) if len(voiced_f0) > 0 else librosa.note_to_hz('C4')
            if tonic == 0:
                tonic = 1.0  # Prevent division by zero

        cents = 1200 * np.log2(np.maximum(f0, 1e-6) / tonic)
        return cents, tonic
    except Exception as e:
        print(f"Error in hz_to_cents for {name}: {e}")
        return np.array([]), None

def remove_transition_regions_scrollable(cents, times, filename, sd_threshold=3, hop_size=1, window_size=3):
    """Detect stable pitch regions with improved stability."""
    try:
        cents = force_1d(cents)
        times = force_1d(times)

        if len(times) < window_size:
            return np.array([]), []

        num_windows = (len(times) - window_size) // hop_size + 1
        stable_regions = []

        for i in range(num_windows):
            start, end = i * hop_size, i * hop_size + window_size
            segment = cents[start:end]
            t_segment = times[start:end]
            segment_voiced = segment[segment > -np.inf]

            if len(segment_voiced) >= 2 and np.std(segment_voiced) < sd_threshold:
                stable_regions.append((t_segment[0], t_segment[-1]))

        # Merge contiguous regions
        contiguous_regions = []
        current = None
        for start_t, end_t in stable_regions:
            if current is None:
                current = [start_t, end_t]
            elif start_t - current[1] <= hop_size * controls["time_step"]:
                current[1] = end_t
            else:
                contiguous_regions.append(tuple(current))
                current = [start_t, end_t]

        if current:
            contiguous_regions.append(tuple(current))

        mask = np.zeros_like(times, dtype=bool)
        for start, end in contiguous_regions:
            mask |= (times >= start) & (times <= end)

        return cents[mask], contiguous_regions
    except Exception as e:
        print(f"Error in remove_transition_regions_scrollable for {filename}: {e}")
        return np.array([]), []

### **Cell 5: Feature Extraction Functions**

This cell contains the functions for computing the three main features: Pitch Class Distribution (PCD), Pitch Interval Histogram (PIH), and Note Duration Histogram (NDH).

In [ ]:
def compute_pitch_class_distribution_gaussian(cents, bins=12, sigma=25.0):
    """Compute PCD with gaussian weighting and robust handling."""
    try:
        cents = force_1d(cents)
        wrapped = np.mod(cents[~np.isinf(cents)], 1200)
        if len(wrapped) == 0:
            return np.zeros(bins)

        bin_centers = np.linspace(0, 1200, bins, endpoint=False)
        hist = np.zeros(bins)

        for pitch in wrapped:
            diffs = np.abs(pitch - bin_centers)
            diffs = np.minimum(diffs, 1200 - diffs)
            weights = np.exp(-(diffs**2) / (2 * sigma**2))
            hist += weights / np.sum(weights)

        return hist / np.sum(hist) if np.sum(hist) > 0 else hist
    except Exception as e:
        print(f"Error in compute_pitch_class_distribution_gaussian: {e}")
        return np.zeros(bins)

def compute_pitch_interval_histogram(cents, bins=24, max_interval=1200):
    """Compute PIH with proper array handling."""
    try:
        cents = force_1d(cents)
        if len(cents) < 2:
            return np.zeros(bins)

        intervals = np.diff(cents[~np.isinf(cents)])
        abs_intervals = np.clip(np.abs(intervals), 0, max_interval)
        hist, _ = np.histogram(abs_intervals, bins=bins, range=(0, max_interval))
        return hist / np.sum(hist) if np.sum(hist) > 0 else hist
    except Exception as e:
        print(f"Error in compute_pitch_interval_histogram: {e}")
        return np.zeros(bins)

def compute_note_duration_histogram(contiguous_regions, bins=6, max_duration=3.0):
    """Compute NDH with proper edge case handling."""
    try:
        if not contiguous_regions or len(contiguous_regions) == 0:
            return np.zeros(bins)

        durations = np.array([end - start for start, end in contiguous_regions])
        durations = np.clip(durations, 0, max_duration)
        hist, _ = np.histogram(durations, bins=bins, range=(0, max_duration))
        return hist / np.sum(hist) if np.sum(hist) > 0 else hist
    except Exception as e:
        print(f"Error in compute_note_duration_histogram: {e}")
        return np.zeros(bins)

### **Cell 6: Visualization Function**

This cell contains the `plot_2d_embedding` function, which is a key part of the analysis.

In [ ]:
def plot_2d_embedding(features, labels=None, method='pca', title='2D Projection', show_text=True):
    """Visualize features in 2D space using PCA or t-SNE."""
    try:
        if len(features) < 2:
            print("Not enough data points for visualization (need at least 2)")
            return

        # Standardize features
        features_std = StandardScaler().fit_transform(features)

        # Perform dimensionality reduction
        if method.lower() == 'pca':
            reducer = PCA(n_components=2)
        elif method.lower() == 'tsne':
            reducer = TSNE(n_components=2, perplexity=min(5, len(features)-1), init='pca')
        else:
            raise ValueError("Method must be either 'pca' or 'tsne'")

        emb = reducer.fit_transform(features_std)

        # Create plot
        plt.figure(figsize=(10, 8))
        colors = [get_cmap('tab20' if len(emb) <= 20 else 'hsv')(i % 20) for i in range(len(emb))]
        scatter = plt.scatter(emb[:, 0], emb[:, 1], c=colors, s=100, alpha=0.7)

        # Add labels if requested
        if show_text and labels is not None:
            texts = []
            for i, (x, y) in enumerate(emb):
                texts.append(plt.text(x, y, labels[i], fontsize=9, ha='center', va='center'))
            adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

        plt.title(f'{method.upper()} Projection: {title}', pad=20)
        plt.xlabel('Component 1')
        plt.ylabel('Component 2')
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"Error in visualization: {e}")

### **Cell 7: Main Processing Loop**

This is the main execution block that orchestrates the entire process: iterating through files, calling the functions, and handling errors.

In [ ]:
# --- Main Processing Loop ---
def main():
    features, labels, valid_files = [], [], []

    for filename in song_list:
        try:
            print(f"\nProcessing {filename}...")
            path = os.path.join(sep_audio_dir, filename)

            # Step 1: Pitch extraction
            f0, times, _ = audio_to_sine_wave(path)
            if len(f0) == 0:
                print("Skipping - no pitch data extracted")
                continue

            # Step 2: Convert to cents
            name = filename.split('.')[0][:-7] if filename.endswith('.wav') else filename.split('.')[0]
            cents, tonic = hz_to_cents(f0, name, final_df)
            if len(cents) == 0:
                print("Skipping - no valid cents conversion")
                continue

            # Step 3: Stable region detection
            cents_filtered, contiguous_regions = remove_transition_regions_scrollable(
                cents, times, filename
            )
            if len(cents_filtered) == 0:
                print("Skipping - no stable regions found")
                continue

            # Step 4: Feature extraction
            hist_pcd = compute_pitch_class_distribution_gaussian(cents_filtered)
            hist_pih = compute_pitch_interval_histogram(cents_filtered)
            hist_ndh = compute_note_duration_histogram(contiguous_regions)

            # Ensure all features are 1D arrays of correct length
            hist_pcd = force_1d(hist_pcd, 12)
            hist_pih = force_1d(hist_pih, 24)
            hist_ndh = force_1d(hist_ndh, 6)

            # Combine features
            combined_features = np.concatenate([hist_pcd, hist_pih, hist_ndh])
            features.append(combined_features)
            labels.append(filename.split('_')[0])
            valid_files.append(filename)

            print(f"Successfully processed. Feature shape: {combined_features.shape}")

        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue

    if features:
        features = np.array(features)
        print(f"\nProcessing complete. Final feature matrix shape: {features.shape}")
        print(f"Processed {len(valid_files)}/{len(song_list)} files successfully")

        # Create DataFrame with descriptive column names
        columns = (
            [f"pcd_bin_{i}" for i in range(12)] +
            [f"pih_bin_{i}" for i in range(24)] +
            [f"ndh_bin_{i}" for i in range(6)]
        )
        df_features = pd.DataFrame(features, columns=columns)
        df_features['filename'] = valid_files
        df_features['label'] = labels

        # Save results
        df_features.to_csv(output_csv, index=False)
        print(f"Features saved to {output_csv}")

        # Visualization
        plot_2d_embedding(features, labels, method='pca', title='PCD + PIH + NDH Embedding')
    else:
        print("No valid features extracted from any files")

if __name__ == "__main__":
    main()